In [ ]:
###################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import gc
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import maximum_filter
from scipy.interpolate import RegularGridInterpolator

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#Importing PlottingModelData Class
sys.path.append(os.path.join(mainCodeDirectory,"Functions",'Classes'))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="EulerianReconstruction",
                                dtype='float32',codeSection = "Project_Algorithms")

#data manager class (for saving data)
DataManager_TrackedProfiles_TrackedAscentTrajectories = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=(5 if ModelData.Ntime==133
          else 10 if ModelData.Ntime==221
          else 30 if ModelData.Ntime==661
          else 1)
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
##############################################
#DATA LOADING AND POST-PROCESSING FUNCTIONS
dataName='UpdraftArea' #Run This Code Only After Running "Tracked_Ascent_Trajectories" for dataName='UpdraftArea'

In [ ]:
#Loading SBF X Locations

def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

SBF_XLocations = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)

In [ ]:
#Data Loading and Post-Processing Functions
def LoadAndProcessData(subsetting_RightOfSBF=False,subsetting_time=None,dataName=dataName):
    parcelTypesList = [['CL','nonCL']]; parcelTypes = parcelTypesList[0]
    trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles_TrackedAscentTrajectories, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    priorToAscentArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles_TrackedAscentTrajectories, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')
    if dataName == 'Variables':
        SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary)
    RemoveLastValidTimestep(trajectoriesArrayDictionary)  # trim CU exit timestep before subsetting
    if dataName == 'Variables':
        SubtractSBF_XLocation(trajectoriesArrayDictionary,priorToAscentArrayDictionary,SBF_XLocations)
        SetZeroMassFluxToNaN(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    if dataName == 'Entrainment':
        FixDetrainmentSign(trajectoriesArrayDictionary); FixDetrainmentSign(priorToAscentArrayDictionary)
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)
    if dataName == 'Eulerian_Entrainment':
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)

    if dataName == 'Variables':
        AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary) #for residence time

    #subsetting only right of SBF
    if subsetting_RightOfSBF:
        trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
        LimitLeftOrRightOfSBF(trajectoriesArrayDictionary,trackedArrays,leftRight='right')
        LimitLeftOrRightOfSBF(priorToAscentArrayDictionary,trackedArrays,leftRight='right')
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting time range
    if subsetting_time is not None:
        SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,subsetting_time)
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting supersets
    for arrayDictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='SBF')
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='ColdPool')
        
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def LimitLeftOrRightOfSBF(dictionary,trackedArrays,
                          leftRight='right'):
    leftRightIdentifier=1 if leftRight=='right' else -1
    
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            leftRightList = trackedArrays[parcelType][parcelDepth][:,4] 
            mask = leftRightList == leftRightIdentifier
            for varName in dictionary[parcelType][parcelDepth]:
                dictionary[parcelType][parcelDepth][varName]=dictionary[parcelType][parcelDepth][varName][:,mask]
    # return dictionary
    
def SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,
               subsetting_time):
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    mask = (times >= (subsetting_time[0] or -np.inf)) & (times <= (subsetting_time[1] or np.inf))
    timeIndices=np.arange(ModelData.Ntime)[mask]
    
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                first_valid = np.argmax(~np.isnan(data_1), axis=0)
                keep_mask = np.isin(first_valid, timeIndices)
                data_1[:, ~keep_mask] = np.nan
                data_2[:, ~keep_mask] = np.nan
                
    # return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    use after applying SubsetTime()
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                keep_mask = np.any(~np.isnan(data_1), axis=0)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = data_1[:, keep_mask]
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = data_2[:, keep_mask]
                
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
        
    DataManager_CalculateMeans = DataManager_Class(mainDirectory, scratchDirectory, ModelData, 
                                                   dataType="CalculateMoreVariables",dataName="CalculateMeans",dtype='float32',
                                                   verbose=False)
    
    [inputDataDirectory,dataName] = CallVariable(ModelData, DataManager_CalculateMeans, 'all_times', 'qv_mean',loadData=False) 
    meanData = DataManager.GetTimestepData_V2(inputDataDirectory, 'all_times', dataName=dataName)
    
    varNames = GetMeanVarNames()
    for varName in varNames:
        varNameMean = varName.lower()+'_mean'
        if varNameMean not in meanData: continue        
        varMean = meanData[varNameMean][:]  # shape (t, Nz)
        for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
            for parcelType in dictionary:
                for parcelDepth in dictionary[parcelType]:
                    d = dictionary[parcelType][parcelDepth]
                    zIndexes = d['Z']
                    validMask = ~np.isnan(zIndexes)
    
                    perturbation = np.full(d[varName].shape, np.nan)
                    tIndexes = np.arange(varMean.shape[0])[:, np.newaxis] * np.ones_like(zIndexes, dtype=int)
                    
                    perturbation[validMask] = d[varName][validMask] - varMean[tIndexes[validMask], zIndexes[validMask].astype(int)]    
                    dictionary[parcelType][parcelDepth][varName+'_perturbation'] = perturbation
                    
    #close data
    meanData.close()

def RemoveLastValidTimestep(trajectoriesArrayDictionary):
    """
    Removes the last non-nan value from each parcel across all variables.
    Sets the last valid timestep to NaN for every parcel and variable.
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            d = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            # Use 'z' as reference to find last valid timestep per parcel
            z = d['z']
            validMask = np.isfinite(z)  # shape (t, p)
            
            # For each parcel, find the index of the last valid timestep
            # argmax on flipped axis gives first True from the end
            last_valid_idx = z.shape[0] - 1 - np.argmax(validMask[::-1, :], axis=0)  # shape (p,)
            ever_valid = validMask.any(axis=0)  # guard against all-nan parcels
            
            # Set last valid timestep to NaN for all variables
            for varName in d:
                for p in np.where(ever_valid)[0]:
                    d[varName][last_valid_idx[p], p] = np.nan

def SubtractSBF_XLocation(trajectoriesArrayDictionary, priorToAscentArrayDictionary, SBF_XLocations):

    SBF_XLocations = np.asarray(SBF_XLocations, dtype=float)

    for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:

                d = dictionary[parcelType][parcelDepth]

                xData = d['X']
                validMask = ~np.isnan(xData)

                tIndexes = np.arange(xData.shape[0])[:, np.newaxis] * np.ones_like(xData, dtype=int)

                sbfForParcels = SBF_XLocations[tIndexes]

                validMask = validMask & ~np.isnan(sbfForParcels)

                xPerturbation = np.full(xData.shape, np.nan)
                xPerturbation[validMask] = xData[validMask] - sbfForParcels[validMask]

                d['X_SBF_Relative'] = xPerturbation

def SetZeroMassFluxToNaN(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    Replace zeros with NaN for VMF_g and VMF_c variables.
    """
    for dictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:
                variables = dictionary[parcelType][parcelDepth]
                for varName in ['VMF_g', 'VMF_c']:
                    if varName in variables:
                        variables[varName] = np.where(variables[varName] == 0,
                                                      np.nan,variables[varName])

def FixDetrainmentSign(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            for varName in dictionary[parcelType][parcelDepth]:
                PROCESSED_string = 'PROCESSED_' if 'PROCESSED_' in varName else ""
                DivideMassFlux_string = '_DivideMassFlux' if '_DivideMassFlux' in varName else ""
                if f"{PROCESSED_string}D{DivideMassFlux_string}_" in varName:
                    dictionary[parcelType][parcelDepth][varName] *= -1

def CalculateNetEntrainment(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            variables = dictionary[parcelType][parcelDepth]
            varNames = list(variables.keys())
            entrainment_varNames = [varName for varName in varNames if "E_" in varName]
            for entrainment_varName in entrainment_varNames:
                detrainment_varName = entrainment_varName.replace('E_', 'D_')
                net_varName = entrainment_varName.replace('E_', 'NET_')
                variables[net_varName] = variables[entrainment_varName] - variables[detrainment_varName]

def AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary):
    
    time = (np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6) * 60
    
    for parcelType in trajectoriesArrayDictionary.keys():
        for parcelDepth in trajectoriesArrayDictionary[parcelType].keys():
            dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            Z = dataDictionary['Z']
            z = dataDictionary['z']
            W = dataDictionary['W']
            qcqi = dataDictionary['QCQI']
            
            masks = {
                'ascent': np.isfinite(Z) & (W > 0.1),
                'CB':     np.isfinite(Z) & (qcqi > 1e-6),
                'CU':     np.isfinite(Z) & (W > 0.5) & (qcqi > 1e-6),
            }
            
            for suffix, mask in masks.items():
                first_idx = np.argmax(mask, axis=0)
                
                # time_at_first = time[first_idx]
                never_met = ~np.any(mask, axis=0) #just in case all nans, skips those cases
                time_at_first = np.where(never_met, np.nan, time[first_idx]) #just in case all nans, skips those cases
                
                time_since = time[:, None] - time_at_first[None, :]
                time_since = np.where(mask, time_since, np.nan)

                dataDictionary[f'z_since_{suffix}'] = np.where(mask, z, np.nan)
                dataDictionary[f'W_since_{suffix}'] = np.where(mask, W, np.nan)
                dataDictionary[f'time_since_{suffix}'] = time_since
                # AddNormalizedTimeSinceAscent(dataDictionary, key, mask) #depreciated

    return trajectoriesArrayDictionary
    

# def AddNormalizedTimeSinceAscent(dataDictionary, timeKey, mask): #depreciated

#     T = dataDictionary[timeKey]          # min
#     zHeight = dataDictionary['z'] / 1e3  # km

#     zMasked = np.where(mask, zHeight, np.nan)
#     zStart = np.nanmin(zMasked, axis=0)

#     dzFromStart = zMasked - zStart[None, :]

#     with np.errstate(invalid='ignore', divide='ignore'):
#         normalizedAge = T / dzFromStart

#     dataDictionary[f'{timeKey}_per_km'] = normalizedAge

def SubsetDataParcelType(arrayDictionary,parcelType_original='CL', parcelType_subset='SBF'):
    """
    In arrayDictionary, the parcelType CL is a superset of SBF and ColdPool.
    This function allows simple "isin" mask to create new parcelTypes for the subset of each superset,
    using indicies marking each subset within trackedArrays (loaded below), as calculated from the SubsetParcels code.
    """

    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class,verbose=False)
    
    original = trackedArrays[parcelType_original]['ALL']
    subset = trackedArrays[parcelType_subset]['ALL']
    
    mask = np.isin(original[:,0], subset[:,0])
    
    arrayDictionary[parcelType_subset] = {}
    
    for inner_key1 in arrayDictionary[parcelType_original]:
        arrayDictionary[parcelType_subset][inner_key1] = {}
        for inner_key2 in arrayDictionary[parcelType_original][inner_key1]:
            varData = arrayDictionary[parcelType_original][inner_key1][inner_key2]
            arrayDictionary[parcelType_subset][inner_key1][inner_key2] = varData[:, mask]
    
    return arrayDictionary

In [ ]:
#Loading Data
subsetting_RightOfSBF=False; #subsetting_RightOfSBF=True
subsetting_time=None; #subsetting_time=(12,None)

[trajectoriesArrayDictionary,priorToAscentArrayDictionary]\
=LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time)

In [ ]:
##############################################
#FUNCTIONS

In [ ]:
##############################################
#CALCULATING FUNCTIONS

In [ ]:
#Calculating Functions
def GetCoordinateData(trajectoriesArrayDictionary,parcelType='CL'):
    Z=trajectoriesArrayDictionary[parcelType]['ALL']['Z']
    Y=trajectoriesArrayDictionary[parcelType]['ALL']['Y']
    X=trajectoriesArrayDictionary[parcelType]['ALL']['X']
    return Z,Y,X
def parcelCount(Z_t, Y_t, X_t):
    """
    Z_t, Y_t, X_t: 1D arrays of indices (one entry per parcel)
    Returns:
        count_grid: (Nz, Ny, Nx) array of parcel counts per cell
    """
    #Setup
    Nz=ModelData.Nzh; Ny=ModelData.Nyh; Nx=ModelData.Nxh

    #Removing Nans
    valid = ~(np.isnan(Z_t) | np.isnan(Y_t) | np.isnan(X_t))
    Z_t_valid = Z_t[valid].astype(np.int64)
    Y_t_valid = Y_t[valid].astype(np.int64)
    X_t_valid = X_t[valid].astype(np.int64)
    
    countArray = np.zeros((Nz, Ny, Nx), dtype=np.int32)
    np.add.at(countArray, (Z_t_valid,Y_t_valid,X_t_valid), 1)
    return countArray

def GetSubsettedData(t,countArray,varName='theta_e',
                     expand_kms_horiz=1,expand_index_vert=0): #expand_kms=0 disables maximum_filter for corresponding direction
    var=CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName=varName)
    mask = countArray != 0

    if expand_kms_horiz > 0:
        size_horiz = 2 * int(round(expand_kms_horiz * ModelData.kms)) + 1
        mask = maximum_filter(mask.astype(np.uint8),size=(1, size_horiz, size_horiz)) > 0 #expand horizontally
        
    if expand_index_vert > 0:
        size_vert = 2 * expand_index_vert + 1
        mask = maximum_filter(mask.astype(np.uint8),size=(size_vert, 1, 1)) > 0 #expand vertically

    varArray = np.where(mask, var, np.nan)
    return varArray,var

def MakeCalculations(t,
                    Z,Y,X,varName='theta_e',
                     isGetVarArray=True,
                     expand_kms_horiz=2,expand_index_vert=0): #expand_kms_horiz=1
    Z_t=Z[t]; Y_t=Y[t]; X_t=X[t]
    countArray = parcelCount(Z_t, Y_t, X_t)

    
    if isGetVarArray:
        [varArray,originalVarArray] = GetSubsettedData(t,countArray,varName=varName,
                                    expand_kms_horiz=expand_kms_horiz,expand_index_vert=expand_index_vert)
        
        return [countArray,varArray,originalVarArray]
    else:
        return countArray

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Colorbar Functions
def MakeTheta_e_cmap():
    def band(c1, c2, n):
        return list(mcolors.LinearSegmentedColormap.from_list("tmp", [c1, c2], N=n)(np.linspace(0, 1, n)))

    colors = (
        band("#d9d9d9", "#666666", 11) +   # white -> gray, 11 shades
        band("#9fd89f", "#3f8f3f", 3)  +   # green, 3 shades
        band("#cfe8ff", "#7fb8e8", 5)  +   # light blue, 5 shades
        band("#3f7fc0", "#0a1f5f", 6)  +   # dark blue, 6 shades
        band("#fff84f", "#ffd23f", 4)  +   # yellow, 4 shades
        band("#ffae3f", "#ff7a1f", 3)  +   # orange, 3 shades
        band("#e8451f", "#5a0a0f", 8)      # red, 6 shades
    )

    cmap = mcolors.ListedColormap(colors, name="theta_e_cmap")

    #fixed levels
    levels = np.linspace(336, 356, cmap.N + 1) #full range about 327-374
    # levels = np.linspace(345, 365, cmap.N + 1) 
    return [cmap,levels]

def TestPlot_Theta_e_cmap():
    import matplotlib as mpl
    [cmap,_] = MakeTheta_e_cmap()
    
    fig, ax = plt.subplots(figsize=(8, 1))
    norm = mpl.colors.Normalize(vmin=336, vmax=356)
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
    cb.set_label('K')
    plt.show()

In [ ]:
#Plotting Functions (MaxParcelCount)
def PlotMaxParcelCount_YX(countArray, axis,
                          cmap=plt.cm.viridis.copy(),
                          maxCountList=None):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=0)
    cmap.set_under('white')

    maxCount = np.max(maxCountList) if maxCountList is not None else np.nanmax(plotData)
    levels = GetSafeLevels(maxCount)
    cf = axis.contourf(ModelData.xh, ModelData.yh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar=plt.colorbar(cf, cax=cax, label='max # of parcels')
    cbar.set_ticks(np.arange(0, maxCount + 1))

    axis.set_ylim(0, 160); axis.set_xlim(100, 400)
    axis.set_ylabel('y (km)'); axis.set_xlabel('x (km)')
    
def PlotMaxParcelCount_ZY(countArray, axis,
                          cmap=plt.cm.viridis.copy(),
                          maxCountList=None):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=2)
    cmap.set_under('white')

    maxCount = np.max(maxCountList) if maxCountList is not None else np.nanmax(plotData)
    levels = GetSafeLevels(maxCount)
    cf = axis.contourf(ModelData.yh, ModelData.zh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar=plt.colorbar(cf, cax=cax, label='max # of parcels')
    cbar.set_ticks(np.arange(0, maxCount + 1))

    axis.set_ylim(0, 16); axis.set_xlim(np.floor(ModelData.yh[0]),np.ceil(ModelData.yh[-1]))
    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)')
    
def PlotMaxParcelCount_ZX(countArray, axis,
                          cmap=plt.cm.viridis.copy(), 
                          maxCountList=None):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=1)
    cmap.set_under('white')

    maxCount = np.max(maxCountList) if maxCountList is not None else np.nanmax(plotData)
    levels = GetSafeLevels(maxCount)
    cf = axis.contourf(ModelData.xh, ModelData.zh, plotData, levels=levels, cmap=cmap, extend='min')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar=plt.colorbar(cf, cax=cax, label='max # of parcels')
    cbar.set_ticks(np.arange(0, maxCount + 1))

    axis.set_ylim(0, 16); axis.set_xlim(100, 400)
    axis.set_ylabel('z (km)'); axis.set_xlabel('x (km)')
    
def PlotMaxParcelCount_Z(countArray, axis, 
                         maxCountList=None):
    countArray_nan = countArray.astype(np.float64)
    plotData = np.nanmax(countArray_nan, axis=(1, 2))
    axis.plot(plotData, ModelData.zh)

    maxCount = np.max(maxCountList) if maxCountList is not None else np.nanmax(plotData)
    levels = GetSafeLevels(maxCount)
    if levels is not None: axis.set_xlim(0, levels[-1])
    axis.set_ylim(0, 16)
    axis.set_ylabel('z (km)'); axis.set_xlabel('max # of parcels')

    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cax.axis('off')

def GetSafeLevels(maxCount):
    if not np.isfinite(maxCount) or maxCount <= 0: return None
    return np.arange(0.5, maxCount + 1.5, 1)
    
def PlotMaxParcelCount(countArray,t=None,
                       cmap=plt.cm.turbo.copy(),maxCountList=None):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    PlotMaxParcelCount_YX(countArray, axes[0, 0], cmap=cmap,
                          maxCountList=maxCountList)
    PlotMaxParcelCount_ZY(countArray, axes[0, 1], cmap=cmap,
                          maxCountList=maxCountList)
    PlotMaxParcelCount_ZX(countArray, axes[1, 0], cmap=cmap,
                          maxCountList=maxCountList)
    PlotMaxParcelCount_Z(countArray, axes[1, 1],
                         maxCountList=maxCountList)

    if t is not None: fig.suptitle(f'Max Parcel Counts at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)
    plt.tight_layout()
    return fig

In [ ]:
#Plotting Functions (Variable)    
def PlotVariableAverage_YX(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=0)

    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
    
    if plotType=='contourf':
        cf = axis.contourf(ModelData.xh, ModelData.yh, plotData, levels=levels, cmap=cmap,extend='both')
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.xh, ModelData.yh, plotData, cmap=cmap, vmin=vmin, vmax=vmax)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 160); axis.set_xlim(100, 400)
    axis.set_ylabel('y (km)'); axis.set_xlabel('x (km)')
    
def PlotVariableAverage_ZY(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=2)
    
    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
        
    if plotType=='contourf':
        cf = axis.contourf(ModelData.yh, ModelData.zh, plotData, levels=levels, cmap=cmap,extend='both')
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.yh, ModelData.zh, plotData, cmap=cmap, vmin=vmin, vmax=vmax)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 16); axis.set_xlim(np.floor(ModelData.yh[0]), np.ceil(ModelData.yh[-1]))
    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)')
    
def PlotVariableAverage_ZX(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                           plotType='pcolormesh',cmap='turbo',levels=None):
    plotData = TakeMean(variableArray,axis=1)
    
    if levels is None:
        vmin, vmax = np.nanmin(plotData), np.nanmax(plotData)
        levels = np.linspace(vmin, vmax, 21)
    else:
        vmin=levels[0];vmax=levels[-1]
        
    if plotType=='contourf':
        cf = axis.contourf(ModelData.xh, ModelData.zh, plotData, levels=levels, cmap=cmap,extend='both')
    elif plotType=='pcolormesh':
        cf = axis.pcolormesh(ModelData.xh, ModelData.zh, plotData, cmap=cmap, norm=norm)
        
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    plt.colorbar(cf, cax=cax, label=f"{label} {units}")
    axis.set_ylim(0, 16); axis.set_xlim(100, 400)
    axis.set_ylabel('z (km)'); axis.set_xlabel('x (km)')
    
def PlotVariableAverage_Z(variableArray, axis, label=rf'$\theta_e$ (K)',units=f'(K)',
                          levels=None):
    plotData = TakeMean(variableArray,axis=(1,2))
    axis.plot(plotData, ModelData.zh)

    if levels is not None:
        axis.set_xlim(levels[0],levels[-1])
        axis.set_xticks(levels[::4])
        
    axis.set_ylim(0, 16)
    axis.set_xlabel(f"{label} {units}"); axis.set_ylabel('z (km)')
    divider = make_axes_locatable(axis)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cax.axis('off')
    
def PlotVariableAverage(variableArray,t=None, label=rf'$\theta_e$',units=f'(K)', plotType='contourf'):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    [cmap,levels] = MakeTheta_e_cmap()
    
    PlotVariableAverage_YX(variableArray, axes[0, 0], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotVariableAverage_ZY(variableArray, axes[0, 1], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotVariableAverage_ZX(variableArray, axes[1, 0], label,units, plotType=plotType, cmap=cmap,levels=levels)
    PlotVariableAverage_Z(variableArray, axes[1, 1], label,units, levels=levels)

    if t is not None: fig.suptitle(f'Average {label} at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)
    plt.tight_layout()
    return fig

def GetMaskedVariable(countArray,variableArray):
    mask = countArray.copy() != 0
    variableMasked = np.where(mask, variableArray, np.nan)
    return variableMasked

def TakeMean(var,axis):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Mean of empty slice')
        varMean=np.nanmean(var, axis=axis)
    return varMean

In [ ]:
#Plotting Functions (CrossSections)
def PlotVariableAverage_CrossSection(varArray, xIndices, yIndices,
                       t=None,zlim=(0, 16),useCbarLimits=True,label=rf'$\theta_e$',units=f'(K)'):
    x, y, z = ModelData.xh, ModelData.yh, ModelData.zh

    theta_e_ZX = ExtractCrossSection(varArray, axis=1, idx_spec=yIndices)  # (z, x), collapsed over y
    theta_e_ZY = ExtractCrossSection(varArray, axis=2, idx_spec=xIndices)  # (z, y), collapsed over x

    x_loc = ResolveLocValue(x, xIndices)
    y_loc = ResolveLocValue(y, yIndices)

    # xIndices -> zooms West-East panel's x-axis; yIndices -> zooms South-North panel's x-axis (y-coordinate)
    xlim = ResolveLimFromIndices(x, xIndices) or (x.min(), x.max())
    ylim = ResolveLimFromIndices(y, yIndices) or (y.min(), y.max())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    if t is not None: fig.suptitle(f'Average {label} at {ModelData.time_hrs[t]:.2f} LT', fontsize=14)

    if useCbarLimits:
        [cmap,levels] = MakeTheta_e_cmap()
        norm = mcolors.BoundaryNorm(levels, cmap.N)

    # --- West-East (Z vs X) ---
    ax = axes[0]
    plotData=theta_e_ZX
    if not useCbarLimits:
        # levels = np.linspace(np.nanmin(plotData), np.nanmax(plotData), cmap.N + 1)  # cmap.N + 1 edges -> cmap.N bins
        plotData_masked = SubsetByLims(plotData, x, z, xlim, zlim) #added
        # levels = np.linspace(np.nanmin(plotData_masked), np.nanmax(plotData_masked), cmap.N + 1) #added
        levels=GetIntegerLevels(plotData_masked) #added
        cmap = plt.get_cmap('turbo', len(levels) + 1) #added
        norm = mcolors.BoundaryNorm(levels, cmap.N, extend='both')
    cf = ax.contourf(x, z, plotData, levels=levels, cmap=cmap, norm=norm, extend='both')
    
    # ax.axvline(x_loc, color='black', linestyle='--', linewidth=1.5)
    ax.set_title('West-East Vertical Cross-Sectional Average')
    ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
    ax.set_xlim(xlim)

    # --- South-North (Z vs Y) ---
    ax = axes[1]
    plotData=theta_e_ZY
    if not useCbarLimits:
        # levels = np.linspace(np.nanmin(plotData), np.nanmax(plotData), cmap.N + 1)  # cmap.N + 1 edges -> cmap.N bins
        plotData_masked = SubsetByLims(plotData, y, z, ylim, zlim) #added
        # levels = np.linspace(np.nanmin(plotData_masked), np.nanmax(plotData_masked), cmap.N + 1) #added
        levels=GetIntegerLevels(plotData_masked) #added
        cmap = plt.get_cmap('turbo', len(levels) + 1) #added
        norm = mcolors.BoundaryNorm(levels, cmap.N, extend='both')
    ax.contourf(y, z, plotData, levels=levels, cmap=cmap, norm=norm, extend='both')

    # ax.axvline(y_loc, color='black', linestyle='--', linewidth=1.5)
    ax.set_title('South-North Vertical Cross-Sectional Average')
    ax.set_xlabel('y (km)')
    ax.set_xlim(ylim)

    for ax in axes:
        ax.set_ylim(zlim)

    fig.subplots_adjust(bottom=0.22)
    cax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
    cbar = fig.colorbar(cf, cax=cax, orientation='horizontal', ticks=levels[::2])
    cbar.set_label(f'{label} {units}')
    return fig

def ExtractCrossSection(varArray, axis, idx_spec):
    """
    varArray: 3D array (z, y, x)
    axis: which axis to collapse (1 for y, 2 for x)
    idx_spec: int (single index) or (start, stop) tuple (range -> averaged)
    """
    idx = ResolveIndexSpecification(idx_spec)
    sl = [slice(None)] * varArray.ndim
    sl[axis] = idx
    sliced = varArray[tuple(sl)]

    if isinstance(idx, slice):
        return TakeMean(sliced, axis=axis)
    else:
        return sliced  # already collapsed by direct indexing

def ResolveIndexSpecification(idx_spec):
    """Convert an int or (start, stop) tuple into something usable for indexing."""
    if isinstance(idx_spec, tuple):
        return slice(idx_spec[0], idx_spec[1])
    return idx_spec

def ResolveLocValue(coord_array, idx_spec):
    """Get the coordinate value (for the dashed-line marker) for an int or (start, stop) tuple."""
    if isinstance(idx_spec, tuple):
        return np.mean(coord_array[idx_spec[0]:idx_spec[1]])
    return coord_array[idx_spec]

def ResolveLimFromIndices(coord_array, idx_spec):
    """
    Get the (min, max) coordinate range spanned by idx_spec, for use as an axis xlim/ylim.
    If idx_spec is a single int, there's no natural range -> return None (full domain).
    """
    if isinstance(idx_spec, tuple):
        start, stop = idx_spec
        return (coord_array[start], coord_array[stop - 1])
    return None

def SubsetByLims(data2d, coord_x, coord_z, xlim, zlim):
    """
    data2d: 2D array (z, x)
    coord_x, coord_z: 1D coordinate arrays matching data2d's axes
    xlim, zlim: (min, max) tuples to subset by
    """
    xmask = (coord_x >= xlim[0]) & (coord_x <= xlim[1])
    zmask = (coord_z >= zlim[0]) & (coord_z <= zlim[1])
    return data2d[np.ix_(zmask, xmask)]

def GetIntegerLevels(plotData_masked):
    vmin = np.floor(np.nanmin(plotData_masked))
    vmax = np.ceil(np.nanmax(plotData_masked))
    step = max(1, round((vmax - vmin) / 20))
    levels = np.arange(vmin, vmax + step, step)
    return levels

In [ ]:
#Plot Saving Functions

def GetPlottingDirectory(plotFileName):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING","Project_Algorithms",
                                                       DataManager_TrackedProfiles.dataType,DataManager_TrackedProfiles.dataName)
    
    specificPlottingDirectory = os.path.join(plottingDirectory, 
                                            f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def GetPlottingFileName(t, fileName,parcelType):
    plotFileName = f"{parcelType}/{fileName}_{parcelType}/{fileName}_{parcelType}_{ModelData.timeStrings[t]}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName)
    return plottingFileName

def SaveFigure(fig, t, fileName,parcelType,
               verbose=False,dpi=300):
    # def RoundUpToMultiple(n, block=16): return ((n + block - 1) // block) * block
    def GetFigsizeForPixels(width_px, height_px, dpi): return (width_px / dpi, height_px / dpi)
        
    plottingFileName = GetPlottingFileName(t, fileName,parcelType)
    if verbose: print(f"Saving figure to {plottingFileName}")
    os.makedirs(os.path.dirname(plottingFileName), exist_ok=True)

    figsize = GetFigsizeForPixels(4176, 2368, dpi=dpi)
    fig.set_size_inches(*figsize)
    fig.savefig(plottingFileName, dpi=dpi)#, bbox_inches='tight')
    plt.close(fig)

In [ ]:
##############################################
#ANALYSIS

In [ ]:
##############################################
#CALCULATING

In [ ]:
#Calculating
[Z_CL,Y_CL,X_CL]=GetCoordinateData(trajectoriesArrayDictionary,parcelType='CL')
[Z_CP,Y_CP,X_CP]=GetCoordinateData(trajectoriesArrayDictionary,parcelType='ColdPool')
[Z_nonCL,Y_nonCL,X_nonCL]=GetCoordinateData(trajectoriesArrayDictionary,parcelType='nonCL')

In [ ]:
##############################################
#PLOTTING ANIMATION

In [ ]:
#Plotting Helper Function
def CalculateMaxCountList(Z, Y, X):
    return [np.nanmax(MakeCalculations(t, Z, Y, X, isGetVarArray=False)) 
            for t in tqdm(range(ModelData.Ntime))]

In [ ]:
##############################################
#PLOTTING
plotting=True #Keep True When Job Array is Running
# plotting=False

In [ ]:
loop_elements=[np.abs(ModelData.time_hrs-14).argmin()] #*testing

In [ ]:
#Plotting
if plotting:
    hourTenIndex = np.abs(ModelData.time_hrs-10).argmin()
    # maxCountList = LoadOrRun(CalculateMaxCountList,
    #                          fileDirectory=DataManager_TrackedProfiles.outputDataDirectory,
    #                          fileName="maxCountList_CL.pkl",
    #                          args=(Z_CL, Y_CL, X_CL))
    for t in tqdm(loop_elements):
        if t<hourTenIndex: continue
        
        [countArray_CL,varArray_CL,_]=MakeCalculations(t,Z_CL,Y_CL,X_CL,varName='theta_e')
        fig1 = PlotMaxParcelCount(countArray_CL,t=t, maxCountList=None)
        fig2 = PlotVariableAverage(varArray_CL,t=t, label=rf'$\theta_e$',units=f'(K)')
        fig3 = PlotVariableAverage_CrossSection(varArray_CL,t=t, xIndices=(150,250), yIndices=(50,150), label=rf'$\theta_e$',units=f'(K)')
    
        SaveFigure(fig1, t, fileName='MaxParcelCount',parcelType='CL')
        SaveFigure(fig2, t, fileName='VariableAverage',parcelType='CL')
        SaveFigure(fig3, t, fileName='VariableAverage_CrossSection',parcelType='CL')

In [ ]:
#Plotting
if plotting:
    # maxCountList = LoadOrRun(CalculateMaxCountList,
    #                          fileDirectory=DataManager_TrackedProfiles.outputDataDirectory,
    #                          fileName="maxCountList_nonCL.pkl",
    #                          args=(Z_nonCL, Y_nonCL, X_nonCL))
    for t in tqdm(loop_elements):
        if t<hourTenIndex: continue
            
        [countArray_nonCL,varArray_nonCL,_]=MakeCalculations(t,Z_nonCL,Y_nonCL,X_nonCL,varName='theta_e')
        fig1 = PlotMaxParcelCount(countArray_nonCL,t=t,maxCountList=None)
        fig2 = PlotVariableAverage(varArray_nonCL,t=t, label=rf'$\theta_e$',units=f'(K)')
        fig3 = PlotVariableAverage_CrossSection(varArray_nonCL,t=t, xIndices=(250,350), yIndices=(50,150), label=rf'$\theta_e$',units=f'(K)')
    
        SaveFigure(fig1, t, fileName='MaxParcelCount',parcelType='nonCL')
        SaveFigure(fig2, t, fileName='VariableAverage',parcelType='nonCL')
        SaveFigure(fig3, t, fileName='VariableAverage_CrossSection',parcelType='nonCL')

In [ ]:
##############################################
#ANIMATION FUNCTIONS
animating=True
animating=False

In [ ]:
#Functions
import imageio.v2 as imageio
def MakeAnimation(plottingFilePaths,outputFilePath,fps=3,quality=1,verbose=True):
    with imageio.get_writer(f'{outputFilePath}', fps=fps, codec='libx264', quality=quality) as writer:
        for f in tqdm(plottingFilePaths, desc="Building animation"):
            img = imageio.imread(f)
            writer.append_data(img)
            del img
        gc.collect()
    if verbose: print(f'Saved to {outputFilePath}')

def GetMP4Path(plottingFileName):
    directory = os.path.dirname(plottingFileName)
    parentDirectory = os.path.dirname(directory)

    mp4Name = os.path.basename(directory) + ".mp4"

    return os.path.join(parentDirectory, mp4Name)

In [ ]:
##############################################
#ANIMATION

In [ ]:
#Animating
if animating:
    for parcelType in ['CL','nonCL']:
        for fileName in ['MaxParcelCount','VariableAverage','VariableAverage_CrossSection']:
        
            plottingFilePaths = [GetPlottingFileName(t, fileName,parcelType) for t in np.arange(ModelData.Ntime) if t>=hourTenIndex]
            plottingFileName = GetPlottingFileName(t=0,fileName=fileName,parcelType=parcelType)
            MP4Path=GetMP4Path(plottingFileName)
            MakeAnimation(plottingFilePaths,outputFilePath=MP4Path)

In [ ]:
##############################################
#CALCULATING ORGANIZATION METRICS
plotting=True 
plotting=False #Keep False When Job Array is Running

In [ ]:
##############################################
#CALCULATING FUNCTIONS

In [ ]:
# Loading Giobiagioli's Organization Indices Calculation Functions
# https://github.com/giobiagioli/organization_indices/blob/main/organization_indices.py

# Citations:
#     Biagioli and Tompkins (2023) https://doi.org/10.1175/JAS-D-23-0103.1: code origin
#     Mandorli and Stubenrauch (2023) https://doi.org/10.5194/gmd-17-7795-2024: explanations of each metric

# Metric Descriptions:

# I_org:

# I_org measures convective organization by comparing the observed cumulative distribution of nearest-neighbor distances between convective objects to the theoretical Weibull distribution expected for randomly placed points. Values above 0.5 indicate clustering, near 0.5 indicate randomness, and below 0.5 indicate regularity (Weger et al. 1992; Tompkins and Semie 2017).

# L_org:

# L_org measures convective organization across all spatial scales by computing the normalized integral of the difference between the observed Besag's L-function (Besag, 1977), a variance-stabilizing transformation of Ripley's K-function (Ripley 1976), which counts the mean number of neighboring objects within a radius r, normalized by object density, and its theoretical expectation under complete spatial randomness, where the theoretical value equals r for a random point pattern. Positive values indicate clustering, near zero indicates randomness, and negative values indicate regularity (Biagioli and Tompkins 2023).


sys.path.append('Organization_Indices_Functions/')
from organization_indices import _compute_organization_indices

In [ ]:
def ComputeOrganizationIndices_AllTimes_AllLevels(Z,Y,X,
                                                  ts=range(ModelData.Ntime)):

    [Iorg_List,Lorg_List]=InitializeStorageArrays() #initiating output array

    for t in tqdm(ts):
        #getting masked array for updraft locations
        [countArray, _, _] = MakeCalculations(t, Z, Y, X, varName='winterp', expand_kms_horiz=0,expand_index_vert=0)
        maskArray = np.where(countArray > 0, 1, 0).astype(int)

        #calculate organization across all levels
        [Iorg_List,Lorg_List]=ComputeOrganizationIndices_AllLevels(maskArray,t,
                                                                   Iorg_List,Lorg_List)

    return [Iorg_List,Lorg_List]

def ComputeOrganizationIndices_AllLevels(maskArray,t,  
                                         Iorg_List,Lorg_List,
                                         zLevels=range(ModelData.Nzh)):
    
    for zLevel in zLevels:
        maskArray_slice = maskArray[zLevel]

        numberConvective = maskArray_slice.sum()
        if numberConvective < 2:
            Iorg_List[t, zLevel] = np.nan; Lorg_List[t, zLevel] = np.nan
            continue
    
        result=ComputeOrganizationIndices(maskArray_slice,rmax,dxy)
        
        Iorg_List[t,zLevel]=result['I_org'] # >>0.5 = organized, <=0.5 = random
        Lorg_List[t,zLevel]=result['L_org'] # >0 = more clustered than random, <0 more random than clustered

    return [Iorg_List,Lorg_List]

def ComputeOrganizationIndices(maskArray_slice,rmax,dxy):

    result = _compute_organization_indices({
        "dxy": dxy,
        "cnv_idx": maskArray_slice.T, #tranposing to fix BCs
        "rmax": rmax,
        "bins": np.arange(0, rmax+dxy, dxy),
        "periodic_BCs": False,   
        "periodic_zonal": True, #transpose is periodic in x, open in y
        "clustering_algo": False, # don't merge adjacent convective pixels into objects
    })

    return result

def InitializeStorageArrays():

    Iorg_List=np.zeros((ModelData.Ntime,ModelData.Nzh)); Lorg_List=np.zeros((ModelData.Ntime,ModelData.Nzh))

    return [Iorg_List,Lorg_List]

def GetParameters():
    dxy=ModelData.dx; Lx=dxy*ModelData.Nxh; Ly=dxy*ModelData.Nyh
    rmax = min(Lx, Ly) / 2
    return rmax,dxy
[rmax,dxy]=GetParameters()

In [ ]:
##############################################
#CALCULATING

In [ ]:
#Calculating
if plotting:
    [Iorg_List_CL, Lorg_List_CL] = LoadOrRun(
        function=ComputeOrganizationIndices_AllTimes_AllLevels,
        fileDirectory=DataManager_TrackedProfiles.outputDataDirectory,
        fileName="OrganizationIndices_CL.pkl",
        args=(Z_CL, Y_CL, X_CL)
    )

    [Iorg_List_nonCL, Lorg_List_nonCL] = LoadOrRun(
        function=ComputeOrganizationIndices_AllTimes_AllLevels,
        fileDirectory=DataManager_TrackedProfiles.outputDataDirectory,
        fileName="OrganizationIndices_nonCL.pkl",
        args=(Z_nonCL, Y_nonCL, X_nonCL)
    )

In [ ]:
#################
#############################
#PLOTTING FUNCTIONS

In [ ]:
def MakeOrganizationIndices_ContourPlot(org_List,varName="I_org",
                                vmin=0, vmax=1, step=0.1, vcenter=0.5, extend=None,
                                axis=None,title=None,xLabel=False):
    if axis is None:
        fig, axis = plt.subplots(figsize=(12, 5))
    
    # Iorg_List shape is (ntime, nz)
    cf = axis.contourf(ModelData.time_hrs, ModelData.zh, org_List.T, 
                     levels=np.linspace(vmin, vmax, 20), cmap='RdBu_r', extend=extend)
    # cbar=axis.contour(ModelData.time_hrs, ModelData.zh, org_List.T,
    #            levels=[vcenter], colors='black', linestyles='-', linewidths=1.5)
    cbar = plt.colorbar(cf, ticks=np.arange(vmin, vmax+step, step))
    cbar.set_label(varName)
    
    
    axis.set_ylim(0, 16); axis.set_xlim(left=10)
    axis.set_ylabel('z (km)')
    if xLabel: axis.set_xlabel('time (hrs)')

    if title is not None:
        axis.set_title(title)

def MakeOrganizationIndices_LinePlot(org_List, varName="I_org",
                                     zTarget=(2,4), axis=None, title=None,
                                     xLabel=False, yLim=None, yStep=None,
                                     hline_value=0.5):
    if axis is None:
        fig, axis = plt.subplots(figsize=(8, 4))

    if isinstance(zTarget, tuple):
        zMin, zMax = zTarget
        zMask = (ModelData.zh >= zMin) & (ModelData.zh <= zMax)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            org_TimeSeries = np.nanmean(org_List[:, zMask], axis=1)
        zLabel = f"{zMin}-{zMax} km"

    else:
        zIndex = np.abs(ModelData.zh - zTarget).argmin()
        zActual = ModelData.zh[zIndex]

        org_TimeSeries = org_List[:, zIndex]
        zLabel = f"{zActual:.2f} km"

    axis.plot(ModelData.time_hrs, org_TimeSeries, linewidth=2)
    axis.axhline(hline_value, color="gray", linestyle="--")

    axis.set_xlim(left=10)
    axis.set_ylabel(varName)

    if xLabel:
        axis.set_xlabel("time (hrs)")

    axis.set_title(title if title is not None else f"{varName} at z = {zLabel}")

    if yLim is not None:
        axis.set_ylim(yLim)

    if yLim is not None and yStep is not None:
        axis.set_yticks(np.arange(yLim[0], yLim[1] + yStep, yStep))

    return axis

In [ ]:
#Plotting Saving Functions
def GetPlottingDirectory(plotFileName, parcelType, extension='pdf'):
    def GetPlotType(dataName):
        plotType=f"Project_Algorithms/Tracked_Profiles/EulerianReconstruction"
        return plotType
    def GetFolderName(parcelType):
        folderName=f"{parcelType}/"
        return folderName
        
    plotType = GetPlotType(dataName)
    folderName = GetFolderName(parcelType)

    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    simStr = f"{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz"
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, f"Simulation_{simStr}", folderName)
    os.makedirs(specificPlottingDirectory, exist_ok=True)
    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName+f'.pdf')

    print(f'Saving to: {plottingFileName}')
    return plottingFileName
def SaveFigureToPDF(pdf,fig,plottingFileName):
    pdf.savefig(fig); plt.close(fig)

In [ ]:
##############################################
#PLOTTING

In [ ]:
#Plotting
if plotting:
    plottingFileName = GetPlottingDirectory(plotFileName=f"OrganizationIndices", parcelType='CL_vs_nonCL')
    with PdfPages(plottingFileName) as pdf:

        plotInfo = [
            (Iorg_List_CL,    "I_org", "CL"),
            (Lorg_List_CL,    "L_org", "CL"),
            (Iorg_List_nonCL, "I_org", "nonCL"),
            (Lorg_List_nonCL, "L_org", "nonCL"),
        ]

        contourSettings = {
            "I_org": dict(vmin=0, vmax=1, step=0.1, vcenter=0.5, extend=None),
            "L_org": dict(vmin=-1.5, vmax=1.5, step=0.25, vcenter=0, extend="both"),
        }

        lineSettings = {
            "I_org": dict(yLim=(0, 1), yStep=0.1, hline_value=0.5),
            "L_org": dict(yLim=(-1.5, 1.5), yStep=0.25, hline_value=0),
        }

        fig1, axes1 = plt.subplots(2, 2, figsize=(12, 10), sharey=False,sharex=True, 
                                   gridspec_kw={"wspace": 0.15, "hspace": 0.14})
        axes1 = axes1.ravel()

        for i, (axis, (data, varName, title)) in enumerate(zip(axes1, plotInfo)):
            MakeOrganizationIndices_ContourPlot(
                data,varName=varName,
                axis=axis,title=title,xLabel=(i >= 2),
                **contourSettings[varName]
            )

        fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10), sharey=False,sharex=True,
                                   gridspec_kw={"wspace": 0.17, "hspace": 0.14})
        axes2 = axes2.ravel()

        for i, (axis, (data, varName, title)) in enumerate(zip(axes2, plotInfo)):
            MakeOrganizationIndices_LinePlot(
                data,varName=varName,zTarget=(2,4),
                axis=axis,title=title,xLabel=(i >= 2),
                **lineSettings[varName]
            )

        SaveFigureToPDF(pdf, fig1, plottingFileName)
        SaveFigureToPDF(pdf, fig2, plottingFileName)

In [ ]:
##############################################
#PLOTTING SLICES (mainly for plume vs bubble analysis ==> depreciated)
plotting=True 
plotting=False #Keep False When Job Array is Running

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Functions
def MakeSlicePlot_MultipleTimes(varName,label,unit,
                                Z,Y,X,
                                tlim,zlim,ylim,xlim,
                                isShowFlowVectors=True,
                                expand_index_vert=95):

    #Configure Times
    # tList = list(range(tlim[0], tlim[-1] + 1))
    number_timesteps_every_3_mins = int(3 / (1/ModelData.mins))
    tList = list(range(tlim[0], tlim[-1] + 1, number_timesteps_every_3_mins))
    
    #Create Figure
    ncols = 3
    nrows = int(np.ceil(len(tList) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), sharey=True,sharex=True, squeeze=False)
    axes = axes.flatten()

    # Compute global levels across all timesteps
    allArrays = {}
    for t in tList:
        [_, varArray, _] = MakeCalculations(t, Z, Y, X, varName=varName, expand_index_vert=expand_index_vert)
        allArrays[t] = varArray
    allData = np.concatenate([GetPlotData(a, zlim, ylim, xlim)[0].ravel() for a in allArrays.values()])
    levels = GetLevels(allData, numLevels=21, isSymmetric=CheckIfSymmetric(varName))
    
    # Add to Figure
    for i, t in enumerate(tqdm(tList)):
        varArray = allArrays[t]
    
        MakeSlicePlot(varArray=varArray,label=label,unit=unit,
                      zlim=zlim,ylim=ylim,xlim=xlim,t=t,
                      levels=levels, add_colorbar=False,
                      isSymmetric=CheckIfSymmetric(varName),
                      axis=axes[i],
                      isYLabel=(i % ncols == 0),isXLabel=(i // ncols == nrows - 1))

        if isShowFlowVectors:
            [horizArray, vertArray] = GetFlowArrays(t, Z,Y,X, ylim,expand_index_vert)
            PlotFlowVectors(axes[i], horizArray, vertArray, xlim, ylim, zlim, skip=2)

    # Shared Colorbar
    cmap = 'RdBu_r' if CheckIfSymmetric(varName) else 'turbo'
    norm = BoundaryNorm(levels, ncolors=plt.get_cmap(cmap).N)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=axes.tolist(), label=f'{label} ({unit})', 
                 pad=0.01, fraction=0.02, extend='both')

    return fig

def MakeSlicePlot(varArray,label,unit,
                  zlim,ylim,xlim,t=None, 
                  numLevels=21, levels=None, add_colorbar=True,
                  isSymmetric=False, cmap=None,
                  axis=None, figsize=(8, 6),
                  isYLabel=True,isXLabel=True):
    """
    varArray: 3D array (z, y, x)
    xlim, ylim, zlim: each either a coordinate-value (start, stop) tuple (range,
                       kept as an axis) or a single coordinate value (location to
                       slice/collapse at). Values are resolved against the actual
                       grid coordinates, so this works correctly on stretched grids
                       like ModelData.zh.
    """

    if axis is None:
        fig, axis = plt.subplots(figsize=figsize)
    else:
        fig = axis.get_figure()
    
    [plotData, 
     zSpec,ySpec,xSpec, 
     z_isRange,y_isRange,x_isRange] = GetPlotData(varArray, zlim, ylim, xlim)
    plotData = np.squeeze(plotData)

    coords = {'z': ModelData.zh[zSpec] if z_isRange else None,
              'y': ModelData.yh[ySpec] if y_isRange else None,
              'x': ModelData.xh[xSpec] if x_isRange else None}
    ranges_kept = [k for k, v in coords.items() if v is not None]
    if len(ranges_kept) != 2:
        raise ValueError("Exactly one of xlim/ylim/zlim must be a scalar "
                          f"(got ranges for: {ranges_kept})")

    if 'z' not in ranges_kept:
        coordX, coordY = coords['x'], coords['y']
    elif 'y' not in ranges_kept:
        coordX, coordY = coords['x'], coords['z']
    else:
        coordX, coordY = coords['y'], coords['z']

    if levels is None:
        levels = GetLevels(plotData, numLevels, isSymmetric=isSymmetric)
    if cmap is None:
        cmap = 'RdBu_r' if isSymmetric else 'turbo'

    cf = axis.contourf(coordX, coordY, plotData, levels=levels, cmap=cmap, extend='both')
    if add_colorbar:
        fig.colorbar(cf, ax=axis, label=f'{label} ({unit})')
    axis.set_xlim(coordX.min(), coordX.max())
    axis.set_ylim(coordY.min(), coordY.max())

    if t is not None:
        axis.set_title(f't = {ModelData.time_hrs[t]:.2f} LT')

    if isYLabel: axis.set_ylabel(f'{ranges_kept[0]} (km)')
    if isXLabel: axis.set_xlabel(f'{ranges_kept[1]} (km)') 
    return fig

def GetPlotData(varArray, zlim, ylim, xlim):
    zSpec, z_isRange = FormatSlice(zlim, ModelData.zh)
    ySpec, y_isRange = FormatSlice(ylim, ModelData.yh)
    xSpec, x_isRange = FormatSlice(xlim, ModelData.xh)
    plotData = np.squeeze(varArray[zSpec, ySpec, xSpec])
    return [plotData, 
            zSpec,ySpec,xSpec, 
            z_isRange,y_isRange,x_isRange]

def GetLevels(plotData, numLevels=21, isSymmetric=False):
    if not isSymmetric:
        vmin = np.nanmin(plotData)
        vmax = np.nanmax(plotData)
    else:
        vmax = np.nanmax(np.abs(plotData))
        vmin = -vmax

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        return np.linspace(0, 1, numLevels)  # fallback dummy levels

    return np.linspace(vmin, vmax, numLevels)

def FormatSlice(spec, coord_array):
    """Convert an int/float (coordinate value -> nearest index) or (start, stop)
    coordinate-value tuple into something usable for numpy indexing."""
    if isinstance(spec, tuple):
        start_idx = ResolveScalarToIndex(coord_array, spec[0])
        stop_idx = ResolveScalarToIndex(coord_array, spec[1])
        return slice(start_idx, stop_idx + 1), True   # inclusive stop
    idx = ResolveScalarToIndex(coord_array, spec)
    return idx, False

def ResolveScalarToIndex(coord_array, value):
    """Find the index of the first coord_array entry >= value (handles stretched grids)."""
    idx = np.searchsorted(coord_array, value)
    idx = min(idx, len(coord_array) - 1)  # clamp in case value exceeds the grid's max
    return idx

def CheckIfSymmetric(varName):
    if varName in ['winterp','buoyancy2']:
        return True
    else:
        return False

def PlotFlowVectors(ax, horizArray, vertArray, xlim, ylim, zlim, skip=3, scale=None, color='k', 
                    method=3):
    """
    Overlays flow vectors onto an existing slice plot's axis.
    """
    zSpec, z_isRange = FormatSlice(zlim, ModelData.zh)
    ySpec, y_isRange = FormatSlice(ylim, ModelData.yh)
    xSpec, x_isRange = FormatSlice(xlim, ModelData.xh)

    hData = np.squeeze(horizArray[zSpec, ySpec, xSpec])
    vData = np.squeeze(vertArray[zSpec, ySpec, xSpec])

    coords = {'z': ModelData.zh[zSpec] if z_isRange else None,
              'y': ModelData.yh[ySpec] if y_isRange else None,
              'x': ModelData.xh[xSpec] if x_isRange else None}
    ranges_kept = [k for k, v in coords.items() if v is not None]

    if len(ranges_kept) != 2:
        raise ValueError("Exactly one of xlim/ylim/zlim must be a scalar "
                          f"(got ranges for: {ranges_kept})")

    if 'z' not in ranges_kept:
        coordX, coordY = coords['x'], coords['y']
    elif 'y' not in ranges_kept:
        coordX, coordY = coords['x'], coords['z']
    else:
        coordX, coordY = coords['y'], coords['z']

    if method==1:
        ax.quiver(coordX[::skip], coordY[::skip],
                  hData[::skip, ::skip], vData[::skip, ::skip],
                  scale=scale, color=color)

    elif method==2:
        X_mesh, Y_mesh = np.meshgrid(coordX, coordY)
        ax.quiver(X_mesh[::skip, ::skip], Y_mesh[::skip, ::skip],
                  hData[::skip, ::skip], vData[::skip, ::skip],
                  scale=scale, color=color)

    # elif method == 3:
    #     # Streamplot expects 1D arrays for the grid lines and 2D arrays for the vector components.
    #     ax.streamplot(coordX, coordY, hData, vData, 
    #                   density=1.5, color=color, linewidth=1)

    elif method == 3:            
            # 1. Create a perfectly uniform grid spanning the exact same range
            coordX_uniform = np.linspace(coordX.min(), coordX.max(), len(coordX))
            coordY_uniform = np.linspace(coordY.min(), coordY.max(), len(coordY))
            
            # 2. Set up the interpolator based on your original stretched axes
            interp_h = RegularGridInterpolator((coordY, coordX), hData, bounds_error=False, fill_value=0)
            interp_v = RegularGridInterpolator((coordY, coordX), vData, bounds_error=False, fill_value=0)

            # 3. Build a grid of points to evaluate the interpolation
            X_uni_mesh, Y_uni_mesh = np.meshgrid(coordX_uniform, coordY_uniform)
            interp_points = np.stack([Y_uni_mesh, X_uni_mesh], axis=-1)

            # 4. Generate the uniform data
            hData_uniform = interp_h(interp_points)
            vData_uniform = interp_v(interp_points)
            
            # 5. Plot using the uniform grid setup
            ax.streamplot(coordX_uniform, coordY_uniform, hData_uniform, vData_uniform, 
                          density=1.5, color=color, linewidth=1)
        
def GetFlowArrays(t, Z,Y,X, ylim,
                  expand_index_vert):

    [_, _,vertArray] = MakeCalculations(t, Z, Y, X, varName='winterp', expand_index_vert=expand_index_vert)

    if isinstance(ylim, (int, float)):
        # West-East slice (ylim is the scalar) -> horizontal component is U
        [_, _,horizArray] = MakeCalculations(t, Z, Y, X, varName='uinterp', expand_index_vert=expand_index_vert)
    else:
        # South-North slice (xlim is the scalar) -> horizontal component is V
        [_, _,horizArray] = MakeCalculations(t, Z, Y, X, varName='vinterp', expand_index_vert=expand_index_vert)

    return [horizArray, vertArray]

In [ ]:
##############################################
#PLOTTING

In [ ]:
#Plotting Settings
expand_index_vert=95
varNames=['theta_e','winterp','buoyancy2']
labels = [r'$\theta_e$', r'$w$', r'$B$']
units  = [r'$K$', r'$m\ s^{-1}$', r'$m\ s^{-2}$']


# varNames=['PROCESSED_Entrainment_DivideMassFlux_g','PROCESSED_Detrainment_DivideMassFlux_g']
# # varNames=['PROCESSED_Entrainment_DivideMassFlux_c','PROCESSED_Detrainment_DivideMassFlux_c']
# labels = [f'PROCESSED_E_DivideMassFlux_g', f'PROCESSED_D_DivideMassFlux_g']
# units  = [r'$m^{-1}$', r'$m^{-1}$']

In [ ]:
#Plotting
if plotting:
    #TIMES
    tStart = 198; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
    
    plottingFileName = GetPlottingDirectory(plotFileName=f"CaseStudy_SBF_LEFT_CL_t={tStart}-{tEnd}", parcelType='CL')

    with PdfPages(plottingFileName) as pdf:
        for varName,label,unit in zip(varNames,labels,units):
        
            
            
            
            #XSLICE
            #Set Up Coordinates
            
            zlim=(0, 10); ylim=50; xlim=(180, 200);
            
            fig1 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CL, Y_CL, X_CL,
                                              tlim,zlim,ylim,xlim)
            
            #YSLICE
            zlim=(0, 10); ylim=(46,56); xlim=194;
            
            fig2 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CL, Y_CL, X_CL,
                                              tlim,zlim,ylim,xlim)

            # SaveFigureToPDF(pdf,fig1,plottingFileName)
            # SaveFigureToPDF(pdf,fig2,plottingFileName)

In [ ]:
#Plotting
if plotting:
    #TIMES
    tStart = 172; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
    
    plottingFileName = GetPlottingDirectory(plotFileName=f"CaseStudy_SBF_RIGHT_CL_t={tStart}-{tEnd}", parcelType='CL')

    with PdfPages(plottingFileName) as pdf:
        for varName,label,unit in zip(varNames,labels,units):
            
            #XSLICE
            #Set Up Coordinates
            
            zlim=(0, 10); ylim=100; xlim=(200, 250);
            
            fig1 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CL, Y_CL, X_CL,
                                              tlim,zlim,ylim,xlim)
            
            #YSLICE
            zlim=(0, 10); ylim=(95,105); xlim=209;
            
            fig2 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CL, Y_CL, X_CL,
                                              tlim,zlim,ylim,xlim)

            SaveFigureToPDF(pdf,fig1,plottingFileName)
            SaveFigureToPDF(pdf,fig2,plottingFileName)

In [ ]:
#Plotting
if plotting:
    #TIMES
    tStart = 159; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
    
    plottingFileName = GetPlottingDirectory(plotFileName=f"CaseStudy_CP_CL_t={tStart}-{tEnd}", parcelType='CL')

    with PdfPages(plottingFileName) as pdf:
        for varName,label,unit in zip(varNames,labels,units):
            
            #XSLICE
            #Set Up Coordinates
            
            zlim=(0, 10); ylim=100; xlim=(380, 400);
            
            fig1 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CP, Y_CP, X_CP,
                                              tlim,zlim,ylim,xlim)
            
            #YSLICE
            zlim=(0, 10); ylim=(95,105); xlim=390;
            
            fig2 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_CP, Y_CP, X_CP,
                                              tlim,zlim,ylim,xlim)

            SaveFigureToPDF(pdf,fig1,plottingFileName)
            SaveFigureToPDF(pdf,fig2,plottingFileName)

In [ ]:
#Plotting
if plotting:
    #TIMES
    tStart = 159; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
    
    plottingFileName = GetPlottingDirectory(plotFileName=f"CaseStudy_nonCL_t={tStart}-{tEnd}", parcelType='nonCL')

    with PdfPages(plottingFileName) as pdf:
        for varName,label,unit in zip(varNames,labels,units):
        
            #TIMES 
            tStart = 141; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
            
            #XSLICE
            #Set Up Coordinates
            zlim=(0, 6); ylim=50; xlim=(368, 378);
            
            fig1 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_nonCL, Y_nonCL, X_nonCL,
                                              tlim,zlim,ylim,xlim)
            
            #YSLICE
            #Set Up Coordinates
            tStart = 141; tEnd = tStart-1 + int(18*ModelData.mins); tlim=(tStart,tEnd)
            zlim=(0, 6); ylim=(46,56); xlim=373;
            
            fig2 = MakeSlicePlot_MultipleTimes(varName,label,unit,
                                              Z_nonCL, Y_nonCL, X_nonCL,
                                              tlim,zlim,ylim,xlim)

            SaveFigureToPDF(pdf,fig1,plottingFileName)
            SaveFigureToPDF(pdf,fig2,plottingFileName)